# C02–M2–L2 — Construct a QA Bot that Leverages the LangChain and LLM to Answer Questions from Loaded Document (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original notebook/files](https://github.com/hengkuankian/learning_ibm_rag_agentic/blob/main/Course2_Build_RAG_Applications/Construct_QA_Bot_w_LangChain/qabot.py)

**Before you run:** add your free `GOOGLE_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey)) to Colab 🔑 Secrets and enable notebook access. No GPU needed — a CPU runtime is enough.

**What changed vs. the IBM version**
- Skills Network venv + requirements.txt, the old Groq/HF `get_llm()` setup cell, the watsonx shim and `llm_factory.py` removed → Cell 1 = minimal Gemini install (+ `gradio`, `pypdf`, `langchain-community`, `langchain-text-splitters`, `langchain-classic`), Cell 2 = universal Gemini init.
- `qabot.py` kept as one `%%writefile` cell with the lab's function names: `get_llm()` now returns `ChatGoogleGenerativeAI("gemini-3.8-flash")` (was Mistral Small 3.1 on watsonx); `watsonx_embedding()` returns `GoogleGenerativeAIEmbeddings("models/gemini-embedding-2")` (was all-MiniLM on watsonx).
- `Chroma` from `langchain_chroma`; `RetrievalQA` from `langchain-classic` (the lab teaches it).
- Earlier Colab fixes kept: `PyPDFLoader` accepts the `type="filepath"` string, `flagging_mode="never"`, `launch(share=True, debug=True)` behind `if __name__ == "__main__"` so each task can be tested from the notebook.
- Colab exercises at the end: embedding-2 vs embedding-001, sources in the answer, refusing prompt (legacy + native LCEL version), cached vector DB, multi-PDF.

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai gradio pypdf langchain-community langchain-text-splitters langchain-classic

### 🔑 Gemini initialization
Run once. Creates `llm` (`gemini-3.8-flash`), `embeddings` (`models/gemini-embedding-2`) and the 1-line `llm_model()` alias used by older lab cells. Needs the `GOOGLE_API_KEY` Colab secret.

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")   # Gemini 3: keep the default temperature (1.0); lower values can cause looping
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
# (.text, not .content: Gemini 3.x can return content as a list of blocks; .text is always a str)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

## The project file — `qabot.py`
The lab builds this file function by function: `get_llm` → `document_loader` → `text_splitter` → `vector_database` + `watsonx_embedding` → `retriever` → `retriever_qa` → Gradio interface. Function names are the lab's own; only their bodies now build Gemini objects (`ChatGoogleGenerativeAI`, `GoogleGenerativeAIEmbeddings`) from the `GOOGLE_API_KEY` set in Cell 2.

In [ ]:
%%writefile qabot.py
# Colab (Gemini): ibm_watsonx_ai ModelInference / GenParams / EmbedTextParamsMetaNames / Credentials and
#                 langchain_ibm WatsonxLLM, WatsonxEmbeddings → langchain_google_genai (key from os.environ)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
import uuid
from langchain_classic.chains import RetrievalQA   # LangChain 1.x: legacy chains live in langchain-classic
# from huggingface_hub import HfFolder   # unused, and removed in huggingface_hub 1.x

import gradio as gr

# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## LLM
def get_llm():
    # Colab (Gemini): mistralai/mistral-medium-2505 (lab PDF) / mistral-small-3-1-24b-instruct-2503 (qabot.py) → gemini-3.8-flash
    model_id = 'gemini-3.8-flash'
    parameters = {
        "max_output_tokens": 1024,   # was GenParams.MAX_NEW_TOKENS: 256 (+ headroom for thinking tokens)
        "thinking_level": "low",     # Gemini 3: thinking tokens count toward max_output_tokens
        # "temperature": 0.5,        # was GenParams.TEMPERATURE — Gemini 3: keep the default 1.0
    }
    watsonx_llm = ChatGoogleGenerativeAI(model=model_id, **parameters)   # was WatsonxLLM(model_id, url, project_id, params)
    return watsonx_llm

## Document loader
def document_loader(file):
    # Colab fix: gr.File(type="filepath") passes a str path, older Gradio passed an object with .name
    loader = PyPDFLoader(file if isinstance(file, str) else file.name)
    loaded_document = loader.load()
    return loaded_document

## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=50,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

## Vector db
def vector_database(chunks):
    embedding_model = watsonx_embedding()
    # Colab fix: own collection per call — without collection_name every call appends to Chroma's shared
    # in-memory 'langchain' collection, so chunks pile up and come back duplicated
    vectordb = Chroma.from_documents(chunks, embedding_model, collection_name=f"qabot_{uuid.uuid4().hex[:8]}")
    return vectordb

## Embedding model
def watsonx_embedding():
    # Colab (Gemini): ibm/slate-125m-english-rtrvr-v2 → models/gemini-embedding-2
    # (the function name is kept so the lab's screenshot tasks still match)
    watsonx_embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
    return watsonx_embedding

## Retriever
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()
    return retriever

## QA Chain
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)
    qa = RetrievalQA.from_chain_type(llm=llm, 
                                    chain_type="stuff", 
                                    retriever=retriever_obj, 
                                    return_source_documents=False)
    response = qa.invoke(query)
    return response['result']

# Create Gradio interface
rag_application = gr.Interface(
    fn=retriever_qa,
    flagging_mode="never",  # Gradio 5: was allow_flagging=False
    inputs=[
        gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),  # Drag and drop file upload
        gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
    ],
    outputs=gr.Textbox(label="Output"),
    title="RAG Chatbot",
    description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
)

# Launch the app (guarded so the notebook can import and test each function first)
if __name__ == "__main__":
    rag_application.launch(share=True, debug=True)  # Colab: public link (was server_name="0.0.0.0", server_port=7860)


## Test each step (the lab's screenshot tasks)
The final submission asks for screenshots of each component. Run these cells and capture the output.

Sample document: *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (Lewis et al., 2020). Use any PDF you like.

In [ ]:
!wget -q -O sample.pdf https://arxiv.org/pdf/2005.11401 && ls -lh sample.pdf
import sys, importlib; sys.path.insert(0, ".")
import qabot; importlib.reload(qabot)

In [ ]:
# Task 1 · PDF loader
docs = qabot.document_loader("sample.pdf")
print(len(docs), "pages\n", docs[0].page_content[:1000])

In [ ]:
# Task 2 · Text splitter
chunks = qabot.text_splitter(docs)
print(len(chunks), "chunks\n", chunks[0].page_content)

In [ ]:
# Task 3 · Embedding model
emb = qabot.watsonx_embedding()
vec = emb.embed_query("How are you?")
print(len(vec), vec[:5])

In [ ]:
# Task 4 · Vector database
vdb = qabot.vector_database(chunks)
for d in vdb.similarity_search("What is RAG?", k=3): print("-", d.page_content[:150].replace("\n", " "))

In [ ]:
# Task 5 · Retriever
ret = qabot.retriever("sample.pdf")
for d in ret.invoke("Which datasets are used for open-domain QA?"): print("-", d.page_content[:150].replace("\n", " "))

In [ ]:
# Task 6 · QA chain
print(qabot.retriever_qa("sample.pdf", "What is this paper talking about?"))

## Launch the QA bot
Upload a PDF, type a question, press **Submit**. The cell blocks while the app runs — press ⏹ to stop.

In [ ]:
!python qabot.py

---
# 🧪 Colab practice exercises
Work inside the notebook with `qabot` imported (no relaunch needed). Re-run `!python qabot.py` after editing the file.

> 💡 **How to use the ✅ Answer cells:** each exercise is followed by a hidden **✅ Answer** cell (form view, code collapsed). Try the exercise first. To check, tick `run_answer` in the form (or double-click to show the code and set `run_answer = True`) and run the cell. With `run_answer` unticked, the cell stops with *Answer not run* — so **Runtime → Run all** stops at the first answer cell; run the cells below it one by one.

### C1 · `gemini-embedding-2` vs `gemini-embedding-001`
Swap the embedding model for the older Gemini one and compare retrieved chunks.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
old_db = Chroma.from_documents(chunks, GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"), collection_name="emb001")
q = "What is the difference between RAG-Sequence and RAG-Token?"
for name, db in [("gemini-embedding-2", vdb), ("gemini-embedding-001", old_db)]:
    print(f"\n== {name}"); [print("-", d.page_content[:120].replace("\n", " ")) for d in db.similarity_search(q, k=3)]
# TODO: change watsonx_embedding() in qabot.py to accept a model name and add a Gradio dropdown for it.

In [ ]:
# @title ✅ Answer — C1 · gemini-embedding-2 vs gemini-embedding-001 (dropdown)
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import uuid
import gradio as gr
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA

EMBED_MODELS = ["models/gemini-embedding-2", "models/gemini-embedding-001"]

# 1) watsonx_embedding() with a model name (copy these 2 functions into qabot.py to change the app itself)
def watsonx_embedding(model_name="models/gemini-embedding-2"):
    return GoogleGenerativeAIEmbeddings(model=model_name)

def vector_database(chunks, model_name="models/gemini-embedding-2"):
    return Chroma.from_documents(chunks, watsonx_embedding(model_name),
                                 collection_name=f"c1_{uuid.uuid4().hex[:8]}")   # own collection per call

def retriever_qa(file, query, model_name):
    chunks = qabot.text_splitter(qabot.document_loader(file))
    qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), chain_type="stuff",
                                     retriever=vector_database(chunks, model_name).as_retriever(),
                                     return_source_documents=False)
    return qa.invoke(query)["result"]

# 2) Same question with both models
q = "What is the difference between RAG-Sequence and RAG-Token?"
for m in EMBED_MODELS:
    print(f"== {m}\n{retriever_qa('sample.pdf', q, m)[:400]}\n")

# 3) The lab interface + a dropdown for the embedding model
c1_app = gr.Interface(
    fn=retriever_qa,
    flagging_mode="never",
    inputs=[gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),
            gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here..."),
            gr.Dropdown(EMBED_MODELS, value=EMBED_MODELS[0], label="Embedding model")],
    outputs=gr.Textbox(label="Output"),
    title="RAG Chatbot (embedding picker)")
c1_app.launch(share=True)   # stop later with c1_app.close()

# Expected / what you should see: Two answers (one per embedding model) for the RAG-Sequence vs RAG-Token question, then a Gradio app with an embedding-model dropdown. Retrieved chunks differ slightly between the two models.


### C2 · Show the sources
`retriever_qa` already uses `return_source_documents=True` but throws them away. Return the answer **plus** page numbers.

In [ ]:
from langchain_classic.chains import RetrievalQA
def retriever_qa_with_sources(file, query):
    qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), chain_type="stuff",
                                     retriever=qabot.retriever(file), return_source_documents=True)
    res = qa.invoke(query)
    pages = sorted({d.metadata.get("page", 0) + 1 for d in res["source_documents"]})
    return f"{res['result']}\n\nSources: pages {pages}"
print(retriever_qa_with_sources("sample.pdf", "Which retriever does RAG use?"))
# TODO: wire this function into a gr.Interface with the same inputs as the lab.

In [ ]:
# @title ✅ Answer — C2 · Show the sources (Gradio)
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import gradio as gr
from langchain_classic.chains import RetrievalQA

def retriever_qa_with_sources(file, query):
    qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), chain_type="stuff",
                                     retriever=qabot.retriever(file), return_source_documents=True)
    res = qa.invoke(query)
    pages = sorted({d.metadata.get("page", 0) + 1 for d in res["source_documents"]})   # PyPDFLoader pages are 0-based
    return f"{res['result']}\n\nSources: pages {pages}"

c2_app = gr.Interface(
    fn=retriever_qa_with_sources,
    flagging_mode="never",
    inputs=[gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),
            gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")],
    outputs=gr.Textbox(label="Output"),
    title="RAG Chatbot with sources")
print(retriever_qa_with_sources("sample.pdf", "Which retriever does RAG use?"))
c2_app.launch(share=True)   # stop later with c2_app.close()

# Expected / what you should see: The answer followed by `Sources: pages [2, …]` (1-based page numbers), and an app whose output shows the same text.


### C3 · Custom prompt that refuses to guess

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
PROMPT = PromptTemplate.from_template("""Use only the context to answer. If it is not in the context, say "Not in the document."
{context}
Question: {question}
Answer:""")
qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), retriever=ret, chain_type_kwargs={"prompt": PROMPT})
for q in ["Who are the authors?", "What is the capital of France?"]:
    print(q, "→", qa.invoke(q)["result"])

In [ ]:
# C3b · The same refusing QA bot as a native LCEL chain (no legacy RetrievalQA)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
format_docs = lambda docs: "\n\n".join(d.page_content for d in docs)
lcel_qa = {"context": ret | format_docs, "question": RunnablePassthrough()} | PROMPT | llm | StrOutputParser()
for q in ["Who are the authors?", "What is the capital of France?"]:
    print(q, "→", lcel_qa.invoke(q))

In [ ]:
# @title ✅ Answer — C3 · Refusing prompt inside the app
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

PROMPT = PromptTemplate.from_template("""Use only the context to answer. If it is not in the context, say "Not in the document."
{context}
Question: {question}
Answer:""")

def retriever_qa_strict(file, query):          # drop-in replacement for qabot.retriever_qa
    qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), chain_type="stuff",
                                     retriever=qabot.retriever(file),
                                     chain_type_kwargs={"prompt": PROMPT})   # ← the custom prompt
    return qa.invoke(query)["result"]

for q in ["What does RAG combine with a seq2seq model?",   # answerable from the paper
          "What is the capital of France?"]:             # not in the paper
    print(f"{q}\n  → {retriever_qa_strict('sample.pdf', q)}\n")

# Expected / what you should see: In-document questions are answered; *What is the capital of France?* returns exactly **Not in the document.**


### C4 · Don't re-embed on every question
`retriever_qa` rebuilds the vector DB on every click. Cache it per file path.

In [ ]:
import functools, time
@functools.lru_cache(maxsize=8)
def cached_retriever(path):
    return qabot.retriever(path)
for i in range(2):
    t = time.time(); cached_retriever("sample.pdf"); print(f"call {i+1}: {time.time() - t:.2f}s")
# TODO: use cached_retriever inside retriever_qa and measure the speed-up in the Gradio app.

In [ ]:
# @title ✅ Answer — C4 · Cached retriever in the app
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import functools, time
from langchain_classic.chains import RetrievalQA

@functools.lru_cache(maxsize=8)
def cached_retriever(path):                     # one vector DB per uploaded file path
    return qabot.retriever(path)

def retriever_qa_cached(file, query):
    qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), chain_type="stuff",
                                     retriever=cached_retriever(file), return_source_documents=False)
    return qa.invoke(query)["result"]

cached_retriever.cache_clear()
for fn in (qabot.retriever_qa, retriever_qa_cached):
    times = []
    for q in ["What is RAG?", "Which datasets are used?", "Who wrote the paper?"]:
        t = time.time(); fn("sample.pdf", q); times.append(time.time() - t)
    print(f"{fn.__name__:20}", " · ".join(f"{x:.2f}s" for x in times))
print(cached_retriever.cache_info())

# In the app: use fn=retriever_qa_cached in gr.Interface (same inputs/outputs as the lab)

# Expected / what you should see: Uncached: every call ≈ same time (re-embeds). Cached: 1st call slow, next calls ≈ 0.0 s; `CacheInfo(hits=2, misses=1, …)`.


### C5 · Ask across several PDFs
Change the interface to `gr.File(file_count="multiple")` and load every file into one vector store.

In [ ]:
def multi_retriever(paths):
    all_chunks = []
    for p in paths:
        all_chunks += qabot.text_splitter(qabot.document_loader(p))
    return qabot.vector_database(all_chunks).as_retriever(search_kwargs={"k": 5})
# TODO: download a second PDF (e.g. https://arxiv.org/pdf/1706.03762), build multi_retriever([...]) and ask a question
# that needs both papers.

In [ ]:
# @title ✅ Answer — C5 · Ask across several PDFs
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import gradio as gr
from langchain_classic.chains import RetrievalQA

!wget -q -O attention.pdf https://arxiv.org/pdf/1706.03762 && ls -lh attention.pdf

def multi_retriever(paths):
    all_chunks = []
    for p in paths:
        all_chunks += qabot.text_splitter(qabot.document_loader(p))
    return qabot.vector_database(all_chunks).as_retriever(search_kwargs={"k": 5})

def multi_qa(files, query):
    paths = [f if isinstance(f, str) else f.name for f in (files or [])]
    qa = RetrievalQA.from_chain_type(llm=qabot.get_llm(), chain_type="stuff",
                                     retriever=multi_retriever(paths), return_source_documents=True)
    res = qa.invoke(query)
    srcs = sorted({f"{d.metadata['source']} p.{d.metadata.get('page', 0) + 1}" for d in res["source_documents"]})
    return res["result"] + "\n\nSources: " + ", ".join(srcs)

print(multi_qa(["sample.pdf", "attention.pdf"],
               "How do the RAG paper and the Transformer paper (Attention Is All You Need) differ in what they propose?"))

c5_app = gr.Interface(
    fn=multi_qa, flagging_mode="never",
    inputs=[gr.File(label="Upload PDF Files", file_count="multiple", file_types=['.pdf'], type="filepath"),
            gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")],
    outputs=gr.Textbox(label="Output"), title="RAG Chatbot (multiple PDFs)")
c5_app.launch(share=True)   # stop later with c5_app.close()

# Expected / what you should see: `attention.pdf` downloaded, one answer using chunks from both papers (RAG + Transformer), and an app with `file_count="multiple"`.
